In [1]:
# thank you https://www.kaggle.com/ashujoshi23

In [2]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import warnings
import os
# from transformers import logging as hf_logging

# 1. 一般的なPythonのWarning（FutureWarningやDeprecationWarningなど）を消す
warnings.filterwarnings('ignore')

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("hf_token")

# Use the token to log in
from huggingface_hub import login
login(token=huggingface_token)

wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

# Log in to Weights & Biases
if wandb_api_key:
    wandb.login(key=wandb_api_key)
    print("W&B logged in successfully!")
else:
    print("W&B API key not found. Running without experiment tracking.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: udaken10 (udaken10-npo-comhbo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logged in successfully!


In [5]:
# Cell 3: Imports & Configuration
import os
import json
import numpy as np
import pandas as pd
import librosa
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn.functional as F
import imageio
# Config
ARCHIVE_PATH = '/kaggle/input/datasets/andrewmvd/covid19-cough-audio-classification' 
MODEL_ID = 'google/medgemma-1.5-4b-it'
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3  # 　kaggle notebook の制限のために、testでは3だが、本番では20〜30で訓練をする
IMAGE_SIZE = 224
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

2026-02-19 01:08:51.184938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771463331.395164      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771463331.455753      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771463331.980158      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771463331.980189      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771463331.980192      55 computation_placer.cc:177] computation placer alr

Using device: cuda


In [6]:
# Cell 4: Audio Processing & Data Loading
def audio_to_mel_spectrogram(audio_path, target_size=(224, 224)):
    try:
        y, sr = librosa.load(audio_path, sr=None)
    except:
        try:
            reader = imageio.get_reader(audio_path)
            y = np.array([x for x in reader])
            if len(y.shape) > 1: y = y.mean(axis=1)
            y = y.flatten()
            if y.max() > 1.0 or y.min() < -1.0: y = y / 32768.0
            sr = 48000
        except:
            return Image.new('RGB', target_size, 'black')
    
    try:
        melspec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
        melspec_norm = (melspec_db - melspec_db.min()) / (melspec_db.max() - melspec_db.min())
        melspec_img = (melspec_norm * 255).astype(np.uint8)
        melspec_color = cv2.applyColorMap(melspec_img, cv2.COLORMAP_MAGMA)
        melspec_color = cv2.cvtColor(melspec_color, cv2.COLOR_BGR2RGB)
        melspec_resized = cv2.resize(melspec_color, target_size)
        return Image.fromarray(melspec_resized)
    except:
        return Image.new('RGB', target_size, 'black')
def load_data(archive_dir='data'):
    files = os.listdir(archive_dir)
    uuid_map = {}
    for f in files:
        base, ext = os.path.splitext(f)
        if base not in uuid_map: uuid_map[base] = {}
        if ext == '.json': uuid_map[base]['json'] = f
        elif ext in ['.webm', '.ogg', '.wav', '.mp3']: uuid_map[base]['audio'] = f
    
    records = []
    for uuid, item in uuid_map.items():
        if 'json' in item and 'audio' in item:
            try:
                with open(os.path.join(archive_dir, item['json'])) as jf:
                    meta = json.load(jf)
                    status = meta.get('status')
                    if status in ['healthy', 'COVID-19']:
                         records.append({'filepath': os.path.join(archive_dir, item['audio']), 'label': status, 'label_id': 1 if status == 'COVID-19' else 0})
            except: pass
    return pd.DataFrame(records)
df_all = load_data(ARCHIVE_PATH)
print(f'Loaded {len(df_all)} samples.')
# Balance Dataset
covid = df_all[df_all['label']=='COVID-19']
healthy = df_all[df_all['label']=='healthy'].sample(n=len(covid), random_state=42)
df_balanced = pd.concat([covid, healthy]).sample(frac=1).reset_index(drop=True)
print(f'Balanced Dataset: {len(df_balanced)} samples.')

Loaded 13634 samples.
Balanced Dataset: 2310 samples.


In [7]:
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label'])

print(f'Training Samples: {len(train_df)}, Validation Samples: {len(eval_df)}')

Training Samples: 1848, Validation Samples: 462


In [8]:
# Cell 5: Model Setup
print('Loading Gemma 3...')
# 4-bit量子化設定
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type='nf4', 
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map='auto', 
    trust_remote_code=True
)

# --- [FIX 1] Enable Gradient Checkpointing & Input Gradients ---
# これが "element 0 of tensors does not require grad" エラーを修正します
model.gradient_checkpointing_enable() 
model.enable_input_require_grads() 
model.config.use_cache = False # 学習時はCacheをOFFにする

# LoRA Configuration
peft_config = LoraConfig(
    r=8, lora_alpha=16, 
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

class CovidAudioDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = audio_to_mel_spectrogram(row['filepath'])
        
        # --- [FIX 2] Fix Label Integration ---
        # "suffix" が無視される問題を修正。プロンプトと正解ラベルをテキストとして結合します。
        # Old: text="detect covid", suffix="positive" (Ignored)
        # New: text="detect covid Answer: positive"
        
        label_text = 'positive' if row['label']=='COVID-19' else 'negative'
        full_text = self.processor.boi_token + f'detect covid Answer: {label_text}'
        
        inputs = self.processor(
            text=full_text, 
            images=image, 
            return_tensors='pt', 
            padding='max_length', 
            max_length=512, 
            truncation=True
        )
        
        # Create labels (Standard Causal LM training)
        if 'labels' not in inputs:
            inputs['labels'] = inputs['input_ids'].clone()
            
        # Remove batch dimension added by processor
        for k,v in inputs.items(): 
            inputs[k] = v.squeeze(0)
            
        return inputs

# dataset = CovidAudioDataset(df_balanced, processor)
# train_df と eval_df それぞれでデータセットを作る
train_dataset = CovidAudioDataset(train_df, processor)
eval_dataset = CovidAudioDataset(eval_df, processor)

Loading Gemma 3...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

trainable params: 16,394,240 || all params: 4,316,473,712 || trainable%: 0.3798


In [9]:
"""
# Cell 6: Metrics & Training

import torch
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support



# --- 1. GPUメモリのお掃除 ---
# これを入れると、前の実行のゴミが消えてOOM（メモリ不足）になりにくくなります
torch.cuda.empty_cache()

# --- 2. 評価指標の計算関数 ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions[0].argmax(-1) if isinstance(pred.predictions, tuple) else pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# --- 3. Training Arguments (あなたの設定 + 必須設定) ---
'''
training_args = TrainingArguments(
    output_dir='./checkpoints',     # ★必須：保存先
    learning_rate=LEARNING_RATE,    # ★必須：学習率 (Cell 3で定義済み)
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # 安定性重視で増やしましたね（Good!）
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,             # 30から3へ短縮（まずは動作確認としてOK）
    
    # 【注意】KaggleのT4 GPUなら bf16=False, fp16=True の方が安定する場合があります
    # もしエラーが出たらここを fp16=True に戻す
    bf16=True,                      
    
    eval_strategy="epoch",          # クラッシュ回避のためepochごとに変更
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    group_by_length=True,           # 音声の長さがバラバラな時に高速化します
    optim="paged_adamw_8bit",
    logging_steps=10
)

'''


import torch
# Clear VRAM before training
torch.cuda.empty_cache()
training_args = TrainingArguments(
    output_dir='./checkpoints',     # ★必須：保存先
    learning_rate=LEARNING_RATE,    # ★必須：学習率 (Cell 3で定義済み)
    
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # Increased for better stability これは素晴らしい
    gradient_checkpointing=True,
    num_train_epochs=3,             # Changed from 30 to 3 ああ、まあねえ、現実的にはね～
    bf16=True,                      # Native support on T4 for less OOMs
    eval_strategy="epoch",          # Changed from 'steps' to avoid mid-train crashこれは知らなかった
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    group_by_length=True,           # Speeds up processing of variable length audioあ、こんなことができるんだ！
    optim="paged_adamw_8bit"
)


# --- 4. Trainerの作成と実行 ---
trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=train_dataset, 
    eval_dataset=eval_dataset,      # 前の手順で作った評価用データ
    compute_metrics=compute_metrics
)


# 学習開始
trainer.train()

# 保存
model.save_pretrained('final_model')
processor.save_pretrained('final_model')
"""

'\n# Cell 6: Metrics & Training\n\nimport torch\nfrom transformers import TrainingArguments, Trainer\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support\n\n\n\n# --- 1. GPUメモリのお掃除 ---\n# これを入れると、前の実行のゴミが消えてOOM（メモリ不足）になりにくくなります\ntorch.cuda.empty_cache()\n\n# --- 2. 評価指標の計算関数 ---\ndef compute_metrics(pred):\n    labels = pred.label_ids\n    preds = pred.predictions[0].argmax(-1) if isinstance(pred.predictions, tuple) else pred.predictions.argmax(-1)\n    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=\'binary\')\n    acc = accuracy_score(labels, preds)\n    return {\'accuracy\': acc, \'f1\': f1, \'precision\': precision, \'recall\': recall}\n\n# --- 3. Training Arguments (あなたの設定 + 必須設定) ---\n\'\'\'\ntraining_args = TrainingArguments(\n    output_dir=\'./checkpoints\',     # ★必須：保存先\n    learning_rate=LEARNING_RATE,    # ★必須：学習率 (Cell 3で定義済み)\n    \n    per_device_train_batch_size=1,\n    gradient_accumulation_steps=16, # 安定性重視で増

In [ ]:
# Cell 6: Metrics & Training
import torch
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- 1. GPUメモリのお掃除 ---
torch.cuda.empty_cache()

# --- 2. 評価指標を計算する関数 ---
def compute_metrics(eval_pred):
    # preprocess_logits_for_metrics で軽量化したデータ(preds)がここに来ます
    preds, labels = eval_pred
    
    # predsはすでに argmax されているので、そのまま使えます
    # ※-100 (パディングなど) の部分は無視して計算する必要がありますが、
    # 簡易的にそのまま計算、もしくは必要なトークンのみ抽出する処理を入れるのが一般的です。
    # ここではエラー回避を優先し、単純な比較にします。
    
    # 形状を合わせる（バッチサイズ x シーケンス長 になっている場合があるため平坦化）
    preds = preds.flatten()
    labels = labels.flatten()
    
    # ラベルが -100 の場所（学習対象外）は除外する
    mask = labels != -100
    preds = preds[mask]
    labels = labels[mask]
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, 
                                                               # average='binary', 
                                                               average = 'weighted',
                                                               zero_division=0)
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# --- [重要！追加] 確率データを軽量化する関数 ---
def preprocess_logits_for_metrics(logits, labels):
    """
    検証時にGPUメモリがパンクしないように、
    重い確率データ(float)を捨てて、予測した単語ID(int)だけを残す関数
    """
    if isinstance(logits, tuple):
        logits = logits[0]
    # 確率が最大のインデックス（単語ID）だけを返す
    return logits.argmax(dim=-1)

# --- 3. Training Arguments ---
training_args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    num_train_epochs=3,　　# testではkaggle notebook の制限のために3にしたが、本当に訓練するには20~30にする
    learning_rate=LEARNING_RATE,
    
    # T4 GPU向け設定
    bf16=False,
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    
    # --- 評価の設定 ---
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    
    # --- [重要！追加設定] OOM回避のキモ ---
    per_device_eval_batch_size=1,   # 検証時もバッチサイズを1にする
    eval_accumulation_steps=1,      # 溜め込まずにすぐCPUへ逃がす
    
    group_by_length=False,
)

# --- 4. Trainerの作成 ---
trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=train_dataset, 
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    
    # ★これを渡すことで劇的にメモリを節約できます
    preprocess_logits_for_metrics=preprocess_logits_for_metrics 
)

# 学習開始
trainer.train()

# 保存
model.save_pretrained('final_model')
processor.save_pretrained('final_model')

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss


In [ ]:
'''
# VRAMエラー版　VRAMがパンクしなければ、こちらを採用
# Cell 6: Metrics & Training
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- 1. GPUメモリのお掃除 ---
# これを入れると、前の実行のゴミが消えてOOM（メモリ不足）になりにくくなります
torch.cuda.empty_cache()


# --- [追加] 評価指標を計算する関数 ---
def compute_metrics(pred):
    labels = pred.label_ids
    # 確率が一番高いものを予測結果とする
    preds = pred.predictions[0].argmax(-1) if isinstance(pred.predictions, tuple) else pred.predictions.argmax(-1)
    
    # 精度(Accuracy), 適合率(Precision), 再現率(Recall), F1スコアを計算
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# --- Training Arguments ---
training_args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    num_train_epochs=30,             # エポック数を30
    learning_rate=LEARNING_RATE,
    # bf16=True,
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    
    # --- [追加] 評価の設定 ---

    eval_strategy="epoch",           # 定期的にテストする
    # eval_steps=20,                   # 20ステップごとにテスト
    save_strategy="epoch",           # 定期的にモデルを保存
    # save_steps=20,
    load_best_model_at_end=True,     # 最後に一番良かったモデルを読み込む
    metric_for_best_model="f1",      # F1スコアが一番高い時を「ベスト」とする（医療AIでは正解率より重要）
    group_by_length = False,
    # Speeds up processing of variable length audio

    
    # eval_strategy="steps",           # 定期的にテストする
    # eval_steps=20,                   # 20ステップごとにテスト
    # save_strategy="steps",           # 定期的にモデルを保存
    # save_steps=20,
    # load_best_model_at_end=True,     # 最後に「一番成績が良かったモデル」を読み込む
    # metric_for_best_model="loss",    # Lossが一番低いものを選ぶ


)

# Trainerの作成
trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=train_dataset,     # 学習データ (前のステップで train_df から作ったもの)
    eval_dataset=eval_dataset,       # 検証データ (前のステップで eval_df から作ったもの)
    compute_metrics=compute_metrics  # ★ここで計算式を渡します！
)

# 学習開始
trainer.train()

# 保存
model.save_pretrained('final_model')
processor.save_pretrained('final_model')
'''


In [ ]:
"""
# Cell 6: Training
training_args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=10

    # --- ★追加設定: 評価（テスト）を行いながら学習する ---
    evaluation_strategy="steps",     # 定期的にテストする
    eval_steps=20,                   # 20ステップごとにテスト
    save_strategy="steps",           # 定期的にモデルを保存
    save_steps=20,
    load_best_model_at_end=True,     # 最後に「一番成績が良かったモデル」を読み込む
    metric_for_best_model="loss",    # Lossが一番低いものを選ぶ
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=dataset, 
    eval_dataset=eval_dataset
)

trainer.train()
model.save_pretrained('final_model')
processor.save_pretrained('final_model')

"""

In [ ]:
# Cell 7: Gradio Inference App (All-in-One Corrected Version)

# --- 0. 必要なライブラリのインストール (ここに追加しました) ---
# これで "pip install" のセルを削除しても大丈夫です
!pip install -q gradio ffmpeg-python librosa imageio opencv-python
!apt-get install -y ffmpeg

import os
import gradio as gr
import torch
import librosa
import numpy as np
import cv2
import traceback
import gc
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import torch.nn.functional as F

# --- 1. メモリの強制解放 (学習直後のOOM回避) ---
print("Cleaning up memory...")
try:
    # 古いモデル変数が残っていたら消す
    if 'model' in globals(): del model
    if 'trainer' in globals(): del trainer
except:
    pass
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

# --- Configuration ---
MODEL_ID = "google/medgemma-1.5-4b-it" 
ADAPTER_PATH = "./final_model"  # ★修正: 学習時の保存名と一致させました

# --- 2. 前処理関数 (学習時と同じロジックを復元) ---
# アプリ側で精度を落とさないために最も重要です
def audio_to_mel_spectrogram_inference(audio_path, target_size=(224, 224)):
    try:
        # 学習時と同じロード方法
        y, sr = librosa.load(audio_path, sr=None) 
    except:
        return Image.new('RGB', target_size, 'black')
    
    try:
        # 学習時と同じパラメータ (n_mels=128, fmax=8000)
        melspec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
        melspec_norm = (melspec_db - melspec_db.min()) / (melspec_db.max() - melspec_db.min())
        melspec_img = (melspec_norm * 255).astype(np.uint8)
        # カラーマップもMAGMAで統一
        melspec_color = cv2.applyColorMap(melspec_img, cv2.COLORMAP_MAGMA)
        melspec_color = cv2.cvtColor(melspec_color, cv2.COLOR_BGR2RGB)
        melspec_resized = cv2.resize(melspec_color, target_size)
        return Image.fromarray(melspec_resized)
    except:
        return Image.new('RGB', target_size, 'black')

# --- Model Loading ---
def load_model_for_inference():
    print("Loading model for inference...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16 # T4向け設定
    )
    
    try:
        # ベースモデル
        base_model = Gemma3ForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # 学習済みLoRAアダプタの結合
        if os.path.exists(ADAPTER_PATH):
            print(f"Loading LoRA adapter from {ADAPTER_PATH}")
            model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
        else:
            print(f"Warning: Adapter not found at {ADAPTER_PATH}. Running with Base Model.")
            model = base_model
            
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        return model, processor
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# モデルロード実行
model_inf, processor_inf = load_model_for_inference()

# --- Grad-CAM Logic (Visual Explainability) ---
def generate_gradcam(model, input_ids, pixel_values):
    # Vision Tower探索
    def find_vision_tower(module):
        if hasattr(module, "vision_tower"): return module.vision_tower
        for child in module.children():
            res = find_vision_tower(child)
            if res: return res
        return None

    vision_tower = find_vision_tower(model)
    if vision_tower is None: return np.zeros((14, 14))
        
    # ターゲット層の特定
    try:
        target_layer = vision_tower.vision_model.encoder.layers[-1].layer_norm1
    except:
        # 構造が違う場合の保険
        target_layer = list(vision_tower.vision_model.encoder.layers)[-1]

    gradients = []
    activations = []
    
    def hook_g(m, i, o): gradients.append(o[0])
    def hook_a(m, i, o): activations.append(o)
    
    h1 = target_layer.register_forward_hook(hook_a)
    h2 = target_layer.register_full_backward_hook(hook_g)
    
    if not pixel_values.requires_grad:
        pixel_values.requires_grad_(True)
    
    model.zero_grad()
    outputs = model(input_ids=input_ids, pixel_values=pixel_values)
    
    score = outputs.logits[0, -1, :].max()
    score.backward()
    
    if gradients and activations:
        grads = gradients[0][0].mean(dim=0)
        acts = activations[0][0]
        cam = torch.matmul(acts, grads)
        
        dim = int(np.sqrt(cam.shape[0]))
        cam = cam.view(dim, dim)
        
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        cam_np = cam.detach().float().cpu().numpy()
    else:
        cam_np = np.zeros((14, 14))
    
    h1.remove(); h2.remove()
    return cam_np

# --- Prediction Function ---
def predict_audio(audio_path):
    if model_inf is None: return "Model Load Error", None
    if audio_path is None: return "Please upload an audio file.", None
    
    try:
        # 1. 前処理 (学習時と同じ関数)
        image = audio_to_mel_spectrogram_inference(audio_path)
        
        # 2. プロンプト (学習時と同じフォーマット)
        prompt = processor_inf.boi_token + "detect covid"
        inputs = processor_inf(text=prompt, images=image, return_tensors="pt", padding="max_length", max_length=512)
        
        input_ids = inputs.input_ids.to(model_inf.device)
        pixel_values = inputs.pixel_values.to(model_inf.device)
        
        # 3. 推論
        with torch.no_grad():
            gen = model_inf.generate(input_ids=input_ids, pixel_values=pixel_values, max_new_tokens=10)
            text_out = processor_inf.batch_decode(gen, skip_special_tokens=True)[0]
        
        # 4. Grad-CAM (説明可能性)
        try:
            cam_map = generate_gradcam(model_inf, input_ids, pixel_values)
            cam_resized = cv2.resize(cam_map, (224, 224))
            cam_resized = np.uint8(255 * cam_resized)
            heatmap = cv2.applyColorMap(cam_resized, cv2.COLORMAP_JET)
            img_np = np.array(image)
            overlay = cv2.addWeighted(img_np, 0.6, cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), 0.4, 0)
        except:
            overlay = np.array(image) # エラー時は画像のみ表示

        return text_out, overlay
        
    except Exception as e:
        return f"Error: {str(e)}", None

# --- UI Launch ---
# サンプルファイルの取得（DataFrameが生きていればそこから取る）
example_files = []
try:
    if 'df_balanced' in globals():
        pos = df_balanced[df_balanced['label']=='COVID-19'].iloc[0]['filepath']
        neg = df_balanced[df_balanced['label']=='healthy'].iloc[0]['filepath']
        example_files = [[pos], [neg]]
except:
    pass

iface = gr.Interface(
    fn=predict_audio,
    inputs=gr.Audio(type="filepath", label="Cough Sound (wav/mp3/webm)"),
    outputs=[gr.Textbox(label="Diagnosis Result"), gr.Image(label="Reasoning (Heatmap)")],
    title="MedGemma COVID-19 Digital Rapid Test",
    description="Upload a cough recording. MedGemma analyzes the sound spectrogram to detect COVID-19 patterns.",
    examples=example_files
)

iface.launch(share=True, debug=True)

In [ ]:
"""

# Cell 7: Inference & Grad-CAM
def generate_gradcam(model, input_ids, pixel_values):
    if hasattr(model, 'base_model'): vision_tower = model.base_model.model.vision_tower
    else: vision_tower = model.model.vision_tower
    target_layer = vision_tower.vision_model.encoder.layers[-1].layer_norm1
    
    gradients = []
    activations = []
    def hook_g(m, i, o): gradients.append(o[0])
    def hook_a(m, i, o): activations.append(o)
    h1 = target_layer.register_forward_hook(hook_a)
    h2 = target_layer.register_full_backward_hook(hook_g)
    
    model.zero_grad()
    outputs = model(input_ids=input_ids, pixel_values=pixel_values)
    score = outputs.logits[0, -1, :].max()
    score.backward()
    
    grads = gradients[0][0].mean(dim=0)
    acts = activations[0][0]
    cam = torch.matmul(acts, grads)
    cam = F.relu(cam.view(int(np.sqrt(cam.shape[0])), -1))
    cam = (cam - cam.min()) / (cam.max() + 1e-8)
    h1.remove(); h2.remove()
    return cam.detach().cpu().numpy()
# Test on a random sample
sample = df_balanced.sample(1).iloc[0]
print(f'Testing on sample: {sample["label"]}')
img = audio_to_mel_spectrogram(sample['filepath'])
prompt = processor.boi_token + 'detect covid'
inputs = processor(text=prompt, images=img, return_tensors='pt', padding='max_length', max_length=512)
input_ids = inputs.input_ids.to(model.device)
pixel_values = inputs.pixel_values.to(model.device)
with torch.no_grad():
    gen = model.generate(input_ids=input_ids, pixel_values=pixel_values, max_new_tokens=10)
    print('Prediction:', processor.batch_decode(gen, skip_special_tokens=True)[0])
cam = generate_gradcam(model, input_ids, pixel_values)
heatmap = cv2.resize(cam, (224, 224))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(np.array(img), 0.6, cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), 0.4, 0)
plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(img); plt.title('Mel Spectrogram')
plt.subplot(1,2,2); plt.imshow(overlay); plt.title('Grad-CAM')
plt.show()

"""

In [ ]:
# !pip install -q gradio transformers peft accelerate bitsandbytes librosa imageio opencv-python ffmpeg-python
# !apt-get install -y ffmpeg

In [ ]:
"""
import os
import gradio as gr
import torch
import librosa
import numpy as np
import cv2
import traceback
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import torch.nn.functional as F
# --- Configuration ---
MODEL_ID = "google/gemma-3-4b-it" 
FINE_TUNED_MODEL_PATH = "./final_medgemma_covid" # Path to your adapter checkpoint
# --- Model Loading ---
def load_model():
    print("Loading model...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    try:
        # Load base model
        model = Gemma3ForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Load LoRA adapter if available
        if os.path.exists(FINE_TUNED_MODEL_PATH):
            print(f"Loading local model from {FINE_TUNED_MODEL_PATH}")
            model = PeftModel.from_pretrained(model, FINE_TUNED_MODEL_PATH)
        else:
            print(f"Fine-tuned model not found at {FINE_TUNED_MODEL_PATH}. Using base model.")
            
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        print("Model loaded successfully.")
        return model, processor
    except Exception as e:
        print(f"Error loading model: {e}")
        traceback.print_exc()
        return None, None
model, processor = load_model()
# --- Audio Preprocessing ---
def audio_to_mel(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    # Normalize
    y = (y - np.mean(y)) / (np.std(y) + 1e-6)
    
    # Compute Mel Spectrogram
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_spect = librosa.power_to_db(mel_spect, ref=np.max)
    
    # Resize to fixed size (e.g., 224x224 for vision encoder)
    mel_spect = cv2.resize(mel_spect, (224, 224))
    
    # Normalize to 0-255 and convert to 3 channels (RGB)
    mel_spect = (mel_spect - mel_spect.min()) / (mel_spect.max() - mel_spect.min()) * 255
    mel_spect = mel_spect.astype(np.uint8)
    mel_spect = cv2.cvtColor(mel_spect, cv2.COLOR_GRAY2RGB)
    
    return mel_spect
    
# --- Grad-CAM Logic ---
def generate_gradcam(model, input_ids, pixel_values):
    # Helper to recursively find vision tower
    def find_vision_tower_recursive(module, depth=0, max_depth=5):
        if depth > max_depth:
            return None
        if hasattr(module, "vision_tower"):
            return module.vision_tower
        
        # Prioritize likely children names
        likely_names = ["base_model", "model", "vision_model", "transformer"]
        for name in likely_names:
            if hasattr(module, name):
                found = find_vision_tower_recursive(getattr(module, name), depth + 1, max_depth)
                if found is not None:
                    return found
        
        return None
    vision_tower = find_vision_tower_recursive(model)
            
    if vision_tower is None:
        print("Could not find vision tower for Grad-CAM.")
        return np.zeros((14, 14)) # Dummy return
        
    target_layer = vision_tower.vision_model.encoder.layers[-1].layer_norm1
    gradients = []
    activations = []
    
    def save_grad(module, grad_input, grad_output):
        gradients.append(grad_output[0])
        
    def save_act(module, input, output):
        activations.append(output)
        
    h1 = target_layer.register_forward_hook(save_act)
    h2 = target_layer.register_full_backward_hook(save_grad)
    
    # Ensure gradients are tracked for inputs to force graph creation (since model is frozen)
    if hasattr(pixel_values, "requires_grad") and not pixel_values.requires_grad:
        pixel_values.requires_grad_(True)
    
    # Forward
    model.zero_grad()
    outputs = model(input_ids=input_ids, pixel_values=pixel_values)
    logits = outputs.logits
    # Target max logit of last token
    score = logits[0, -1, :].max()
    score.backward()
    
    # Compute CAM
    if len(gradients) > 0 and len(activations) > 0:
        grads = gradients[0][0]
        acts = activations[0][0]
        weights = torch.mean(grads, dim=0)
        cam = torch.matmul(acts, weights)
        
        num_patches = cam.shape[0]
        grid = int(np.sqrt(num_patches))
        cam = cam.view(grid, grid)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        # Cast to float32 before numpy conversion to handle bfloat16
        cam_np = cam.detach().float().cpu().numpy()
    else:
         print("Grad-CAM failed: No gradients or activations captured.")
         cam_np = np.zeros((14, 14))
    
    # Cleanup
    h1.remove()
    h2.remove()
    
    return cam_np
# --- Prediction Function ---
def predict_audio(audio_path):
    if model is None:
        return "Model not loaded (CPU/Error)", None
        
    print(f"Processing audio: {audio_path}")
    if audio_path is None:
        return "Please upload an audio file.", None
    if not os.path.exists(audio_path):
        return f"Error: File not found at {audio_path}", None
        
    try:
        # Preprocess
        image = audio_to_mel(audio_path)
        prompt = processor.boi_token + "detect covid"
        inputs = processor(text=prompt, images=image, return_tensors="pt", padding="max_length", truncation=True, max_length=512)
        
        input_ids = inputs.input_ids.to(model.device)
        pixel_values = inputs.pixel_values.to(model.device)
        
        # Inference
        with torch.no_grad():
            generated_ids = model.generate(input_ids=input_ids, pixel_values=pixel_values, max_new_tokens=10)
            result_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Grad-CAM
        try:
            cam_map = generate_gradcam(model, input_ids, pixel_values)
            # Resize CAM to image size
            cam_resized = cv2.resize(cam_map, (224, 224))
            cam_resized = np.uint8(255 * cam_resized)
            heatmap = cv2.applyColorMap(cam_resized, cv2.COLORMAP_JET)
            
            # Overlay
            overlay = cv2.addWeighted(image, 0.6, heatmap, 0.4, 0)
            
            return result_text, overlay
        except Exception as e:
            print(f"Grad-CAM Error: {e}")
            traceback.print_exc()
            return result_text, image # Return original image on CAM error
            
    except Exception as e:
        error_msg = f"Error processing audio:\n{str(e)}\n\nTraceback:\n{traceback.format_exc()}"
        print(error_msg)
        return error_msg, None
# --- Gradio UI ---
iface = gr.Interface(
    fn=predict_audio,
    inputs=gr.Audio(type="filepath", label="Upload Cough Audio (wav/mp3/webm)"),
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Image(label="Grad-CAM Heatmap (Mel Spectrogram)")
    ],
    title="MedGemma COVID-19 Detection",
    description="Upload a cough audio file to detect COVID-19 using the fine-tuned MedGemma model. Displays Grad-CAM heatmap on the Mel Spectrogram.",
    examples=[
        ["/kaggle/input/datasets/andrewmvd/covid19-cough-audio-classification/00014dcc-0f06-4c27-8c7b-737b18a2cf4c.webm"],
        ["/kaggle/input/datasets/andrewmvd/covid19-cough-audio-classification/00014dcc-0f06-4c27-8c7b-737b18a2cf4c.webm"]
    ] # Add example paths if you have them uploaded to Kaggle
)
if __name__ == "__main__":
    iface.launch(share=True) # Share=True enables a public link, useful for Colab/Kaggle
"""

In [ ]:
# Cell 5: Model Setup
"""
print('Loading Gemma 3...')
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
# LoRA Configuration
peft_config = LoraConfig(
    r=8, lora_alpha=16, 
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
class CovidAudioDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = audio_to_mel_spectrogram(row['filepath'])
        prompt = self.processor.boi_token + 'detect covid'
        inputs = self.processor(text=prompt, images=image, suffix='positive' if row['label']=='COVID-19' else 'negative', return_tensors='pt', padding='max_length', max_length=512, truncation=True)
        if 'labels' not in inputs: inputs['labels'] = inputs['input_ids'].clone()
        for k,v in inputs.items(): inputs[k] = v.squeeze(0)
        return inputs
dataset = CovidAudioDataset(df_balanced, processor)

"""

Project Write-up: 

MedGmemma for Covid19

The Digital Antibody

* Title: Every Cough Tells a Story: MedGemma as a Zero-Cost PCR Alternative

English Version

* The Vision: A Lab in Every Pocket

PCR tests are the gold standard, but they are expensive, logistically complex, and often unavailable where they are needed most. We envision a world where a diagnostic lab fits in your pocket.

By leveraging MedGemma, we are creating a low-cost, non-invasive digital screening tool that requires nothing more than a smartphone. This is not just an app; it is a digital rapid test kit that can be distributed instantly to billions of people, bypassing the need for physical supply chains.


* Why MedGemma? The "Medical Native" AI

By feeding massive amounts of medical imagery (spectrograms) into MedGemma, we are waking up a "Medical Native" AI. It brings the encyclopedic knowledge of a specialist to every single diagnosis, spotting patterns in a cough that even a trained ear might miss.

* Our Solution: Bridging the Gap

We treat cough sounds not just as audio, but as biological signals.

* Visual-Textual Fusion: 

We convert audio into Mel-Spectrograms, allowing MedGemma to "see" the pathology.

* Cost-Effective Triage: 

This model serves as a preliminary screening layer. It empowers individuals to make informed decisions—whether to isolate or seek urgent care—without the immediate financial or logistical burden of a PCR test.


* The Impact: Democratizing Expert Diagnostics

We are building a future where the EMR is no longer a passive storage system, but an active diagnostic partner. A future where a smartphone recording in a rural clinic is analyzed with the same rigor as a scan in a top-tier university hospital.
MedGemma is not just a tool; it is the stethoscope of the 21st century. We are turning the world's massive, sleeping medical data into an active force for saving lives, providing a low-cost safety net for everyone, everywhere.

日本語版

タイトル：すべての咳には意味がある：MedGemmaが実現する「ゼロ・コスト」のPCR代替検査 Medgemma Covid19のためのMedGemma

* ビジョン：ポケットの中にある検査室

   
PCR検査は診断のゴールドスタンダードですが、高価であり、物流網が必要で、最も必要とされる場所で利用できないことが多々あります。

私たちのビジョンは、「ポケットに入る検査室」です。 **

私たちはMedGemmaを活用し、スマートフォンさえあれば誰でも利用できる、低コストで非侵襲的なデジタル簡易検査キットを開発しました。これは物理的な試薬や輸送を必要としません。アプリをダウンロードするだけで、瞬時に数十億人に配布可能な、新たなパンデミック対策の防波堤です。

* なぜMedGemmaなのか？ 「医療ネイティブ」なAIの覚醒

MedGemmaは、専門医レベルの知識を持ち、人間の耳では聞き取れない咳の音の特徴から、病理学的なパターンを見つけ出します。

* ソリューション：医療格差を埋める架け橋
   
私たちは、咳の音を単なる音声データではなく、生体シグナルとして扱います。

* 視覚と知識の融合: 音声をメルスペクトログラムに変換し、MedGemmaに「視覚的」に病変を診断させます。

安価なトリアージ（選別）: このモデルは、高価なPCR検査の前段階となる「第0次スクリーニング」として機能します。ユーザーは、経済的・物理的な負担なしに「隔離すべきか？」「病院へ行くべきか？」という重要な判断を、医学的根拠に基づいて即座に行うことができます。

* インパクト：専門医の診断を、すべての人へ

私たちが目指すのは、電子カルテが単なる「保存場所」ではなく、「能動的な診断パートナー」となる未来です。僻地の診療所でスマートフォンを使って録音された音声が、大学病院の検査と同じレベルの厳密さで解析される世界です。

MedGemmaは単なるツールではありません。それは21世紀の「聴診器」そのものです。

私たちは、世界中の電子カルテに眠る膨大なデータを、人命を救うためのアクティブな力へと変えていきます。
